In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.document_loaders import TextLoader
from langchain_core.prompts import PromptTemplate
from langgraph.graph import StateGraph, START, END

#### Document Loading

In [2]:
loader = TextLoader("/home/tacktile/SIM_SupportBot/knowledge_base/SIM_Knowledge_Base_v3.md")
docs = loader.load()

In [3]:
print(docs[0].page_content)

# Simple Invoice Manager (SIM) — Knowledge Base

> **Purpose:** This is a pure reference knowledge base for the Simple Invoice Manager app. It contains feature descriptions, field definitions, business concepts, alternate terminology, and contextual background. Step-by-step instructions are handled separately in the prompt layer.

---

## Table of Contents

1. [Company / Business Setup](#1-company--business-setup)
2. [Clients & Suppliers](#2-clients--suppliers)
3. [Products & Services](#3-products--services)
4. [Invoices](#4-invoices)
5. [Estimates / Proforma Invoices](#5-estimates--proforma-invoices)
6. [Sale Orders](#6-sale-orders)
7. [Purchase Orders](#7-purchase-orders)
8. [Purchase Records](#8-purchase-records)
9. [Inventory Management](#9-inventory-management)
10. [Sale Returns, Refunds & Credit Notes](#10-sale-returns-refunds--credit-notes)
11. [Sub Users & Approvals](#11-sub-users--approvals)
12. [GST Compliance — India](#12-gst-compliance--india)

---

## 1. Company / Business

#### Chunking

In [13]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, 
                                          chunk_overlap=200,
                                          separators=["\n\n", "\n", " ", "", "#", "##", "###", "####"])
chunks = splitter.create_documents([docs[0].page_content])

In [14]:
len(chunks)

109

In [17]:
chunks[100].page_content

'"module": "settings",\n    "intent": "show_outstanding_on_invoice",\n    "tags": ["outstanding balance", "invoice template", "pending dues", "total due", "client balance"],\n    "related_concepts": ["invoice_statuses", "opening_balance"],\n    "priority": "medium"\n  },\n  {\n    "id": "faq_012",\n    "question": "Can we add the opening balance of clients and suppliers?",\n    "alternate_questions": [\n      "How to set prior balance for client?",\n      "Add existing dues for customer",\n      "Opening balance kaise add kare?",\n      "Set starting balance for supplier",\n      "How to enter balance before using app?"\n    ],'

#### Embedding Generation and Vector Store 

In [18]:
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(chunks, embeddings)

In [19]:
vectorstore.index_to_docstore_id

{0: 'f13c60ea-1289-4ade-af9d-3a6d5832a4e5',
 1: '62894fb5-7171-4b98-8a47-cf750e8dd589',
 2: 'e4040273-fb95-4aa3-9fda-a0d1ca0ecddb',
 3: '3925e18c-4e90-41a1-b707-5a4ffbb1f543',
 4: 'd9193573-dc2f-4243-9869-cf459444167d',
 5: '2c9160c3-a5c6-4de3-9214-8a314e8d637e',
 6: '4a496de5-6b60-4844-871f-604a8d3391fb',
 7: '3368382d-2de5-4d05-98a9-c922e84d3879',
 8: '3038d557-3645-40cc-bfce-ff0ecca42307',
 9: '66f6680c-0440-49d0-a027-a60e6e9242b1',
 10: '2da4dbd7-5219-4907-a017-4f25fdc380f0',
 11: 'b615a42f-8ec6-4bfc-9df6-06fcdbcd6479',
 12: 'e4fe2b80-d5ff-4d53-9c5c-ec3c604963c7',
 13: '43c277f2-0647-4da1-97f3-a592e3596cce',
 14: '90f41cb3-6684-44f9-ae09-99eff5b227ab',
 15: '0378930d-bc1b-49ee-bc32-0c942ba7ad0d',
 16: 'f9bdbb3b-1816-4aa0-8dd4-c2197b9c5d9d',
 17: 'ce00bc28-c0c7-4bcc-94e3-9b6ad3d39bac',
 18: 'de433040-5cc2-44e7-86e3-34f2e60bdbe7',
 19: 'c1d74ec9-3cca-4d57-af52-bb8842a74081',
 20: 'ac50a34d-c365-4c5c-8193-309029328e4e',
 21: '7a95e2e7-6ce0-4428-a840-61151e5711a1',
 22: '5a07222c-29b2-

In [23]:
vectorstore.get_by_ids(["f13c60ea-1289-4ade-af9d-3a6d5832a4e5"])

[Document(id='f13c60ea-1289-4ade-af9d-3a6d5832a4e5', metadata={}, page_content='# Simple Invoice Manager (SIM) — Knowledge Base\n\n> **Purpose:** This is a pure reference knowledge base for the Simple Invoice Manager app. It contains feature descriptions, field definitions, business concepts, alternate terminology, and contextual background. Step-by-step instructions are handled separately in the prompt layer.\n\n---\n\n## Table of Contents\n\n1. [Company / Business Setup](#1-company--business-setup)\n2. [Clients & Suppliers](#2-clients--suppliers)\n3. [Products & Services](#3-products--services)\n4. [Invoices](#4-invoices)\n5. [Estimates / Proforma Invoices](#5-estimates--proforma-invoices)\n6. [Sale Orders](#6-sale-orders)\n7. [Purchase Orders](#7-purchase-orders)\n8. [Purchase Records](#8-purchase-records)\n9. [Inventory Management](#9-inventory-management)\n10. [Sale Returns, Refunds & Credit Notes](#10-sale-returns-refunds--credit-notes)\n11. [Sub Users & Approvals](#11-sub-users-

#### Retrival

In [28]:
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 5})

In [29]:
retriever

VectorStoreRetriever(tags=['FAISS', 'OpenAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7dbf2be5b770>, search_kwargs={'k': 5})

In [31]:
retriever.invoke("What is Invoices?")

[Document(id='de433040-5cc2-44e7-86e3-34f2e60bdbe7', metadata={}, page_content='### Online store:  \nProducts added to the Product List automatically appear in the Online Store\n\n\n### 🎥 Video Tutorials\n- [How to Add Product or Service](https://www.youtube.com/watch?v=EmURS_9WiG4&list=PL9NRmuo_lj22dReggkiHIZRUK-b-4kWKz&index=4)\n- [Batch Upload Multiple Products](https://www.youtube.com/watch?v=-zSq_ioaw8A&list=PL9NRmuo_lj22dReggkiHIZRUK-b-4kWKz&index=5)\n- [Setup Product Categorization](https://www.youtube.com/watch?v=Lyt9zfVnRHM&list=PL9NRmuo_lj22dReggkiHIZRUK-b-4kWKz&index=23)\n\n---\n\n## 4. Invoices\n\n### What It Is\nAn invoice is a formal, legally binding document issued by a seller to a buyer, requesting payment for goods delivered or services rendered. It details what was sold, the quantities, prices, applicable taxes, and the total amount due. In India, a GST-registered business must issue a **tax invoice** for all taxable supplies.'),
 Document(id='c1d74ec9-3cca-4d57-af52-

#### Augumentation

In [33]:
llm = ChatOpenAI(model = "gpt-4o-mini", temperature=0.5)

In [34]:
prompt = PromptTemplate(
    template= """You are a helpful assistant for answering questions about the SIM product. Use the following retrieved documents to answer the question. If you don't know the answer, say you don't know. \n\n {context} \n\n Question: {question} \n Answer:""",
    input_variables=["context", "question"]
)

In [35]:
question = "How to create an Invoice?"
retrieved_docs = retriever.invoke(question)

In [36]:
retrieved_docs

[Document(id='c1d74ec9-3cca-4d57-af52-bb8842a74081', metadata={}, page_content='### When & Who\nUsed by any business that sells goods or services — freelancers, retailers, wholesalers, manufacturers, service providers. An invoice is raised after a sale is made or a service is delivered.\n\n---\n\n### 🔁 Flow\n\n1. From the dashboard, **swipe right to left** to find the **Invoice** section.\n2. Tap **Add New Invoice** Set the invoice date. Optionally add a due date.\n3. Tap **Client Name** → select an existing client or create a new one inline.\n4. Tap **Add Items** → select products/services or add new ones on the spot.\n5. Apply discount, tax, and shipping charges if applicable.\n6. In the **Payment** section, record any advance or instant payment received.\n7. Optionally customize the invoice (header, footer, signature, image, notes, custom fields).\n8. Tap **Preview** → then **Save**, **Send**, **Print**, or **Change Template**.\n\n---\n\n### Key Fields'),
 Document(id='f622ef1a-875b

In [44]:
context_text = "\n\n".join([doc.page_content for doc in retrieved_docs])
context_text

'### When & Who\nUsed by any business that sells goods or services — freelancers, retailers, wholesalers, manufacturers, service providers. An invoice is raised after a sale is made or a service is delivered.\n\n---\n\n### 🔁 Flow\n\n1. From the dashboard, **swipe right to left** to find the **Invoice** section.\n2. Tap **Add New Invoice** Set the invoice date. Optionally add a due date.\n3. Tap **Client Name** → select an existing client or create a new one inline.\n4. Tap **Add Items** → select products/services or add new ones on the spot.\n5. Apply discount, tax, and shipping charges if applicable.\n6. In the **Payment** section, record any advance or instant payment received.\n7. Optionally customize the invoice (header, footer, signature, image, notes, custom fields).\n8. Tap **Preview** → then **Save**, **Send**, **Print**, or **Change Template**.\n\n---\n\n### Key Fields\n\n```json\n[\n  {\n    "id": "faq_001",\n    "question": "How to create an invoice?",\n    "alternate_questi

In [45]:
final_prompt = prompt.invoke({'context': context_text, 'question': question})

In [46]:
final_prompt

StringPromptValue(text='You are a helpful assistant for answering questions about the SIM product. Use the following retrieved documents to answer the question. If you don\'t know the answer, say you don\'t know. \n\n ### When & Who\nUsed by any business that sells goods or services — freelancers, retailers, wholesalers, manufacturers, service providers. An invoice is raised after a sale is made or a service is delivered.\n\n---\n\n### 🔁 Flow\n\n1. From the dashboard, **swipe right to left** to find the **Invoice** section.\n2. Tap **Add New Invoice** Set the invoice date. Optionally add a due date.\n3. Tap **Client Name** → select an existing client or create a new one inline.\n4. Tap **Add Items** → select products/services or add new ones on the spot.\n5. Apply discount, tax, and shipping charges if applicable.\n6. In the **Payment** section, record any advance or instant payment received.\n7. Optionally customize the invoice (header, footer, signature, image, notes, custom fields).

#### Generation

In [48]:
answer = llm.invoke(final_prompt)
answer.content

'To create an invoice, go to the Invoice section on the dashboard and tap the **Add New Invoice** button. Set the invoice date and optionally add a due date. Then, tap **Client Name** to select an existing client or create a new one. Next, tap **Add Items** to select products or services or add new ones. You can apply any discounts, taxes, and shipping charges if needed. In the **Payment** section, record any advance or instant payment received. Optionally, customize the invoice with a header, footer, signature, image, notes, or custom fields. Finally, tap **Preview** to review the invoice, and then choose to **Save**, **Send**, or **Print** it. The invoice will appear in your invoice list with a payment status of Not Received until payment is collected.'

#### Building Chain

In [54]:
from langchain_core.runnables import RunnableParallel, RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

In [55]:
def format_docs(retrieved_docs):
    return "\n\n".join([docs.page_content for docs in retrieved_docs])

In [62]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [58]:
parallel_chain.invoke("What is Inventory?")

{'content': '---\n\n## 9. Inventory Management\n\n### What It Is\nInventory management in SIM tracks the quantity of physical goods a business holds at any point in time. Stock levels update automatically — increasing when purchases are recorded and decreasing when sales (invoices) are made. The feature also supports manual adjustments and low-stock alerts.\n\n### When & Who\nRelevant only for businesses dealing in physical goods — shops, wholesalers, distributors, manufacturers. Not applicable to pure service businesses. Should be enabled as part of initial app setup.\n\n### What Affects Inventory\n\n| Action | Effect on Inventory |\n|---|---|\n| Invoice (Sale) | ✅ Stock decreases |\n| Purchase Record | ✅ Stock increases |\n| Manual Adjustment | ✅ Increases or decreases |\n| Estimate / Quotation | ❌ No effect |\n| Sale Order | ❌ No effect |\n| Purchase Order | ❌ No effect |\n| Delivery Note | ❌ No effect |\n\n\n---\n\n### 🔁 Flow — Enabling Inventory\n\n### Alternate Terms\n\n| App Ter

In [59]:
parser = StrOutputParser()

In [63]:
main_chain = parallel_chain | prompt | llm | parser

In [66]:
main_chain.invoke("When to use Purchase record")

'You should use a Purchase Record when goods are actually received from a supplier. It is created at the moment the goods arrive, regardless of whether a Purchase Order was raised beforehand. The Purchase Record formally records the inward movement of stock, updates inventory quantities, and tracks the payment owed to the supplier.'